# Feature engineering::

#### Import and setup

In [ ]:
import pandas as pd
import numpy as np
import yaml
import os

os.chdir("..")

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Ready.")

Ready.


#### Load & filter raw data

In [ ]:
filepath = config["data"]["raw_dir"] + config["data"]["lending_club_file"]
df = pd.read_csv(filepath, low_memory=False)

# Keep only completed loans
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()

# Binary target
df["default"] = (df["loan_status"] == "Charged Off").astype(int)

print(f"Rows: {len(df):,}")

Rows: 1,345,310


#### Select relevant columns

In [ ]:
features = [
    "loan_amnt", "int_rate", "installment",
    "annual_inc", "dti", "emp_length",
    "fico_range_low", "fico_range_high",
    "revol_util", "revol_bal",
    "open_acc", "total_acc", "pub_rec",
    "delinq_2yrs", "inq_last_6mths",
    "home_ownership", "purpose", "grade", "term"
]

target = "default"

df = df[features + [target]].copy()
print(f"Shape: {df.shape}")
df.head()

Shape: (1345310, 20)


,loan_amnt,int_rate,installment,annual_inc,dti,emp_length,fico_range_low,fico_range_high,revol_util,revol_bal,open_acc,total_acc,pub_rec,delinq_2yrs,inq_last_6mths,home_ownership,purpose,grade,term,default
0,3600.0,13.99,123.03,55000.0,5.91,10+ years,675.0,679.0,29.7,2765.0,7.0,13.0,0.0,0.0,1.0,MORTGAGE,debt_consolidation,C,36 months,0
1,24700.0,11.99,820.28,65000.0,16.06,10+ years,715.0,719.0,19.2,21470.0,22.0,38.0,0.0,1.0,4.0,MORTGAGE,small_business,C,36 months,0
2,20000.0,10.78,432.66,63000.0,10.78,10+ years,695.0,699.0,56.2,7869.0,6.0,18.0,0.0,0.0,0.0,MORTGAGE,home_improvement,B,60 months,0
4,10400.0,22.45,289.91,104433.0,25.37,3 years,695.0,699.0,64.5,21929.0,12.0,35.0,0.0,1.0,3.0,MORTGAGE,major_purchase,F,60 months,0
5,11950.0,13.44,405.18,34000.0,10.20,4 years,690.0,694.0,68.4,8822.0,5.0,6.0,0.0,0.0,0.0,RENT,debt_consolidation,C,36 months,0


#### Clean numeric columns

In [ ]:
# int_rate and revol_util are already floats in this dataset
# Just convert term and emp_length

# Convert term to integer months
df["term"] = df["term"].str.strip().str.replace(" months", "").astype(int)

# Convert emp_length to numeric
emp_map = {
    "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3,
    "4 years": 4, "5 years": 5, "6 years": 6, "7 years": 7,
    "8 years": 8, "9 years": 9, "10+ years": 10
}
df["emp_length"] = df["emp_length"].map(emp_map)

print("Numeric cleaning done.")
df[["int_rate", "revol_util", "term", "emp_length"]].head()

Numeric cleaning done.


,int_rate,revol_util,term,emp_length
0,13.99,29.7,36,10.0
1,11.99,19.2,36,10.0
2,10.78,56.2,60,10.0
4,22.45,64.5,60,3.0
5,13.44,68.4,36,4.0


#### Nulls

In [ ]:
# Numeric nulls — fill with median
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols.remove("default")

for col in numeric_cols:
    median = df[col].median()
    null_count = df[col].isnull().sum()
    if null_count > 0:
        print(f"{col}: filling {null_count:,} nulls with median {median:.2f}")
    df[col] = df[col].fillna(median)

# Categorical nulls — fill with "Unknown"
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

print("\nNull counts after cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("All nulls handled." if df.isnull().sum().sum() == 0 else "Still has nulls.")

dti: filling 374 nulls with median 17.61
emp_length: filling 78,511 nulls with median 6.00
revol_util: filling 857 nulls with median 52.20
inq_last_6mths: filling 1 nulls with median 0.00

Null counts after cleaning:
Series([], dtype: int64)
All nulls handled.


C:\Users\micro\AppData\Local\Temp\ipykernel_25988\1925449131.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns.tolist()


#### Feature engineering

In [ ]:
# FICO midpoint
df["fico_score"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

# Loan to income ratio
df["loan_to_income"] = df["loan_amnt"] / (df["annual_inc"] + 1)

# Monthly debt burden
df["monthly_debt"] = df["installment"] / (df["annual_inc"] / 12 + 1)

# Credit utilization flag
df["high_utilization"] = (df["revol_util"] > 75).astype(int)

# Derogatory mark flag
df["has_derog"] = (df["pub_rec"] > 0).astype(int)

print("Engineered features added.")
df[["fico_score", "loan_to_income", "monthly_debt", 
    "high_utilization", "has_derog"]].describe()

Engineered features added.


,fico_score,loan_to_income,monthly_debt,high_utilization,has_derog
count,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06
mean,6.981851e+02,4.925749e+00,2.209325e-01,1.989185e-01,1.693922e-01
std,3.185284e+01,3.329680e+02,9.875793e+00,3.991867e-01,3.750981e-01
min,6.270000e+02,1.714285e-04,6.911988e-05,0.000000e+00,0.000000e+00
25%,6.720000e+02,1.246888e-01,4.625803e-02,0.000000e+00,0.000000e+00
50%,6.920000e+02,1.999950e-01,7.220173e-02,0.000000e+00,0.000000e+00
75%,7.120000e+02,2.909038e-01,1.054124e-01,0.000000e+00,0.000000e+00
max,8.475000e+02,4.000000e+04,1.466850e+03,1.000000e+00,1.000000e+00


#### Encode categoricals

In [ ]:
# One-hot encode low-cardinality categoricals
df = pd.get_dummies(df, columns=["home_ownership", "purpose", "grade"], 
                    drop_first=True)

# Drop original term (already numeric)
print(f"Shape after encoding: {df.shape}")
df.head()

Shape after encoding: (1345310, 46)


,loan_amnt,int_rate,installment,annual_inc,dti,emp_length,fico_range_low,fico_range_high,revol_util,revol_bal,...,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G
0,3600.0,13.99,123.03,55000.0,5.91,10.0,675.0,679.0,29.7,2765.0,...,False,False,False,False,False,True,False,False,False,False
1,24700.0,11.99,820.28,65000.0,16.06,10.0,715.0,719.0,19.2,21470.0,...,False,True,False,False,False,True,False,False,False,False
2,20000.0,10.78,432.66,63000.0,10.78,10.0,695.0,699.0,56.2,7869.0,...,False,False,False,False,True,False,False,False,False,False
4,10400.0,22.45,289.91,104433.0,25.37,3.0,695.0,699.0,64.5,21929.0,...,False,False,False,False,False,False,False,False,True,False
5,11950.0,13.44,405.18,34000.0,10.20,4.0,690.0,694.0,68.4,8822.0,...,False,False,False,False,False,True,False,False,False,False


##### Save data:

In [ ]:
output_path = config["data"]["features_dir"] + "features.parquet"
df.to_parquet(output_path, index=False)
print(f"Saved {len(df):,} rows to {output_path}")
print(f"Final feature count: {df.shape[1] - 1}")  # minus target

Saved 1,345,310 rows to data/features/features.parquet
Final feature count: 45
